## P3.a — Vectores TF-IDF

- **Entrada:** `data/final/` (columna `tokens`, ya generada en el KDD; no se retokeniza)
- **Salida:** `data/p3/vectores/` — un vector TF-IDF normalizado (L2) por proceso, como vector de Spark ML y como arrays `idx`/`val`
- Equivale al `TfidfVectorizer` + coseno de lab3.2, con `HashingTF` + `IDF` de Spark ML. Al normalizar, el coseno es el producto punto

### Imports

In [1]:
import os, socket, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 200)

### Utilidades

- Constantes y funciones compartidas (`03_P3/p3_funciones.ipynb`)

In [2]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
ruta_funciones = str(proyecto / "03_P3" / "p3_funciones.ipynb")
%run $ruta_funciones

final_dir      = proyecto / "data" / "final"
vectores_dir   = proyecto / "data" / "p3" / "vectores"
centroides_dir = proyecto / "data" / "p3" / "centroides"
listas_dir     = proyecto / "data" / "p3" / "listas"
vecinos_dir    = proyecto / "data" / "p3" / "vecinos"

/opt/conda/lib/python3.11/site-packages/nbformat/__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


- Spark

In [3]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName("p3_a_vectores")
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")
           .config("spark.sql.autoBroadcastJoinThreshold", "-1")
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### Verificación del cluster

In [4]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()
print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 0
Cores totales : 2
Memoria driver: 3.0 GB
UI del job    : http://localhost:4040


---
### 1. Carga

In [5]:
assert final_dir.exists(), f"No se encuentra {final_dir}; corre 01_EDA_KDD/KDD/a_transformaciones.ipynb"
procesos = (spark.read.parquet(str(final_dir)).select(*COLS_P3)
                 .where(F.col("anio").isin(ANIOS) & (F.size("tokens") > 0)))
N = procesos.count()
print(f"Procesos con tokens: {N:,}")
procesos.select(F.size("tokens").alias("n")).select(F.min("n"), F.expr("percentile_approx(n, 0.5)").alias("mediana"), F.max("n")).show()
procesos.select("ocid", "categoria", F.slice("tokens", 1, 8).alias("tokens (8 primeros)")).show(3, truncate=70)

Procesos con tokens: 231,171
+------+-------+------+
|min(n)|mediana|max(n)|
+------+-------+------+
|     1|     16|   478|
+------+-------+------+

+---------------------------+---------+----------------------------------------------------------------------+
|                       ocid|categoria|                                                   tokens (8 primeros)|
+---------------------------+---------+----------------------------------------------------------------------+
|ocds-dgv273-seacev3-1085359|    GOODS|[adquisicion, ladrillos, refractarios, manta, fibra, ceramica, refi...|
|ocds-dgv273-seacev3-1090383|    GOODS|[contratacion, bienes, adquisicion, aditivos, concreto, obra, mejor...|
|ocds-dgv273-seacev3-1092010| SERVICES|[contratacion, profesional, servicio, medicina, ocupacional, vigila...|
+---------------------------+---------+----------------------------------------------------------------------+
only showing top 3 rows



### 2. TF-IDF + normalización

In [6]:
t0 = time.time()
vec, idf_model = ajustar_tfidf(procesos)
vec = vec.withColumn("arr", vector_a_arrays("vec")).select("*", "arr.idx", "arr.val").drop("arr")
vec = vec.where(F.size("idx") > 0)
vec.write.mode("overwrite").partitionBy("categoria").parquet(str(vectores_dir))
print(f"Vectores guardados en {time.time() - t0:,.0f} s")
vectores = spark.read.parquet(str(vectores_dir))

Vectores guardados en 44 s


### 3. Checkpoint: parquet de vectores

- Filas ≈ procesos, norma 1, nº de dimensiones activas por vector

In [7]:
n_vec = vectores.count()
chk = (vectores.select(F.size("idx").alias("nnz"), F.aggregate("val", F.lit(0.0), lambda a, x: a + x * x).alias("norma2"))
                .select(F.min("nnz").alias("nnz_min"), F.expr("percentile_approx(nnz, 0.5)").alias("nnz_mediana"), F.max("nnz").alias("nnz_max"),
                        F.min("norma2").alias("norma2_min"), F.max("norma2").alias("norma2_max")).first())
print(f"Vectores: {n_vec:,} (procesos: {N:,}) | dims activas {chk['nnz_min']}–{chk['nnz_max']} (mediana {chk['nnz_mediana']}) | "
      f"norma² {chk['norma2_min']:.4f}–{chk['norma2_max']:.4f} | {len(list(vectores_dir.glob('categoria=*')))} particiones (categoria)")
assert abs(chk["norma2_min"] - 1) < 1e-6 and abs(chk["norma2_max"] - 1) < 1e-6
vectores.groupBy("categoria").count().show()
vectores.select("ocid", F.slice("idx", 1, 5).alias("idx (5)"), F.slice("val", 1, 5).alias("val (5)")).show(3, truncate=60)

Vectores: 231,171 (procesos: 231,171) | dims activas 1–453 (mediana 16) | norma² 1.0000–1.0000 | 3 particiones (categoria)
+---------+------+
|categoria| count|
+---------+------+
| SERVICES|100050|
|    GOODS|104002|
|    WORKS| 27119|
+---------+------+

+---------------------------+------------------------------+------------------------------------------------------------+
|                       ocid|                       idx (5)|                                                     val (5)|
+---------------------------+------------------------------+------------------------------------------------------------+
|ocds-dgv273-seacev3-1092010|    [10, 706, 771, 1848, 2044]|[0.2307835845185198, 0.2894377757731381, 0.35394587037999...|
|ocds-dgv273-seacev3-1092308|  [29, 1960, 3391, 3914, 4007]|[0.5100899480821687, 0.371526355978396, 0.599200216077542...|
|ocds-dgv273-seacev3-1092343|[1221, 1814, 2228, 2375, 2635]|[0.3898801852519337, 0.34212051906716967, 0.2826671174748...|
+----------

### 4. Checkpoint: el coseno nativo coincide con el de Spark ML

- Muestra de pares; `coseno()` (arrays) vs. producto punto de los vectores ML

In [8]:
m = vectores.sample(fraction=min(1.0, 500 / n_vec), seed=SEED).select("ocid", "vec", "idx", "val")
a, b = m.alias("a"), m.alias("b")
dot_ml = F.udf(lambda u, v: float(np.dot(u.toArray(), v.toArray())), "double")
chk2 = (a.join(b, F.col("a.ocid") < F.col("b.ocid")).limit(3000)
         .select(coseno(F.col("a.idx"), F.col("a.val"), F.col("b.idx"), F.col("b.val")).alias("nativo"), dot_ml(F.col("a.vec"), F.col("b.vec")).alias("ml"))
         .select(F.max(F.abs(F.col("nativo") - F.col("ml"))).alias("dif_max"), F.avg("nativo").alias("cos_promedio"), F.count("*").alias("pares")).first())
print(f"Pares: {chk2['pares']:,} | diferencia máxima nativo vs ML: {chk2['dif_max']:.2e} | coseno promedio entre procesos al azar: {chk2['cos_promedio']:.3f}")
assert chk2["dif_max"] < 1e-9

Pares: 3,000 | diferencia máxima nativo vs ML: 5.55e-17 | coseno promedio entre procesos al azar: 0.015


- Siguiente: `b_indice_ivf.ipynb`

### 5. Cierre

In [9]:
spark.stop()